# Packing budget probe

Finds the **maximum sequence length that fits on one H100** during LoRA training for each of the 3 v0 base models, so we can set a fixed per-model packing budget that goes *beyond* the dataset's longest single chain (one long doc + a few short ones share a pack).

It builds the *exact* training-time model by reusing `LoRATrainer.model` (FA2 + LoRA with the right `target_modules` + thinking-token row-scoped grad + `enable_input_require_grads`), enables gradient checkpointing, attaches an AdamW optimizer, then binary-searches the largest length `L` that survives forward + backward + optimizer step.

Budgets differ per model because the cross-entropy logits `[seq_len, vocab]` dominate long-context memory and the vocabs differ a lot (Qwen ~151k, Llama ~128k, Phi-4-mini ~200k).

**This is GPU-only.** Run it on the training box, then copy the recommended budgets into `core/training/packing_budgets.py`.

In [1]:
import gc
from pathlib import Path

import torch
from datasets import Dataset
from transformers import AutoTokenizer

from core.datasets.abstract_dataset_adapter import AbstractDatasetAdapter
from core.training.base_trainer import PackingConfig
from core.training.lora_trainer import (
    LoRASpecificTrainingArgs,
    LoRATrainer,
    LoRATrainerConfig,
    LoRATrainingArgs,
    phi4_mini_lora_target_modules,
)
from core.training.thinking_tokens import setup_thinking_tokens

assert torch.cuda.is_available(), "This probe must run on the GPU box (single H100)."

# Walk up to the repo root (the dir that holds artifacts/).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "artifacts").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
BASE_DIR = REPO_ROOT / "artifacts" / "base_models_v0"

# The 3 main models. phi4_mini fuses q/k/v + gate/up, so it needs custom LoRA targets;
# the other two use the defaults (target_modules=None -> LoRASpecificTrainingArgs default).
MODELS = {
    "qwen_3b": {"target_modules": None},
    "llama_3b": {"target_modules": None},
    "phi4_mini": {"target_modules": phi4_mini_lora_target_modules},
}

# Recommend this fraction of the largest fitting L, to leave headroom for fragmentation,
# multi-segment packs, and torch_compile overhead (the probe runs eager).
SAFETY_MARGIN = 0.90
RAMP_START = 4096       # first length tried / warmup
RAMP_HARD_CAP = 262144  # never probe beyond this
BISECT_TOL = 256        # stop bisection when the OK/OOM gap is this small

TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"repo root: {REPO_ROOT}")
print(f"GPU: {torch.cuda.get_device_name(0)}  total {TOTAL_GIB:.0f} GiB")

repo root: /src/recursive_caft
GPU: NVIDIA H100 PCIe  total 79 GiB


2026-06-24 11:35:06.884 | INFO     | core.utils.logger:<module>:48 - === process start === pid=12367 log_file=/src/recursive_caft/artifacts/logs/eval-20260624-113506-12367.log


In [2]:
class _StubAdapter(AbstractDatasetAdapter):
    """The probe never calls train(); it only needs LoRATrainer.model, which never touches
    the dataset. This stub just satisfies the config type."""

    def process_dataset(self, path_override=None):
        return Dataset.from_dict({"input_ids": [[1]], "labels": [[1]]})

    def save_processed_dataset(self, df, path, tmp): ...


def build_training_model(nick, target_modules):
    """Reproduce the exact training-time model for `nick` on the GPU."""
    model_path = (BASE_DIR / nick).as_posix()

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    setup_thinking_tokens(tokenizer)

    lora_kwargs = {"train_thinking_token_embeddings": True}
    if target_modules is not None:
        lora_kwargs["target_modules"] = target_modules

    cfg = LoRATrainerConfig(
        out_path="/tmp/packing_probe",
        model_id=model_path,
        train_dataset=_StubAdapter(),
        training_args=LoRATrainingArgs(num_train_epochs=1, per_device_train_batch_size=1),
        lora_training_args=LoRASpecificTrainingArgs(**lora_kwargs),
        # budget is the very thing this probe measures. PackingConfig.budget is only read
        # in _prepare_data (train()); for model construction only `packing is not None`
        # matters -- it flips attn_implementation to flash_attention_2. The probe never
        # calls train(), so this placeholder is never used.
        packing=PackingConfig(budget=RAMP_HARD_CAP),
    )
    trainer = LoRATrainer(config=cfg, tokenizer=tokenizer)
    model = trainer.model  # FA2 + LoRA + thinking-token row-scoped grad + enable_input_require_grads
    model.gradient_checkpointing_enable()  # match training_args.gradient_checkpointing=True
    model.train()
    model.cuda()
    return trainer, model, tokenizer

In [3]:
def _make_batch(length, vocab_size):
    # Worst case: a single document of `length` (one attention segment, monotonic
    # position_ids). A short -100 prefix mirrors the masked prompt.
    ids = torch.randint(0, vocab_size, (1, length), device="cuda")
    labels = ids.clone()
    labels[:, : min(8, length)] = -100
    position_ids = torch.arange(length, device="cuda").unsqueeze(0)
    return {"input_ids": ids, "labels": labels, "position_ids": position_ids}


def fits(model, optimizer, length, vocab_size):
    """Run one bs=1 fwd+bwd+step at `length`. Returns (ok, peak_gib)."""
    batch = out = loss = None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    ok, peak = True, 0.0
    try:
        batch = _make_batch(length, vocab_size)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = model(**batch)
            loss = out.loss
        loss.backward()
        optimizer.step()
        peak = torch.cuda.max_memory_allocated() / 2**30
    except torch.cuda.OutOfMemoryError:
        ok = False
    finally:
        optimizer.zero_grad(set_to_none=True)
        del batch, out, loss
        gc.collect()
        torch.cuda.empty_cache()
    return ok, peak


def probe_max_length(model, optimizer, vocab_size):
    # Warmup so AdamW state is allocated before we measure peaks.
    ok, peak = fits(model, optimizer, RAMP_START, vocab_size)
    print(f"  warmup L={RAMP_START}: {'OK' if ok else 'OOM'}" + (f" peak={peak:.1f}GiB" if ok else ""))
    if not ok:
        lo, hi = 256, RAMP_START
    else:
        lo, hi = RAMP_START, RAMP_START * 2
        while hi <= RAMP_HARD_CAP:
            ok, peak = fits(model, optimizer, hi, vocab_size)
            print(f"  ramp   L={hi}: {'OK' if ok else 'OOM'}" + (f" peak={peak:.1f}GiB" if ok else ""))
            if not ok:
                break
            lo, hi = hi, hi * 2
        else:
            print(f"  never OOMed up to hard cap {RAMP_HARD_CAP}")
            return lo
    best = lo
    while hi - lo > BISECT_TOL:
        mid = (lo + hi) // 2
        ok, peak = fits(model, optimizer, mid, vocab_size)
        print(f"  bisect L={mid}: {'OK' if ok else 'OOM'}" + (f" peak={peak:.1f}GiB" if ok else ""))
        if ok:
            best, lo = mid, mid
        else:
            hi = mid
    return best

In [4]:
results = {}
for nick, spec in MODELS.items():
    print(f"\n=== {nick} ===")
    trainer = model = tokenizer = optimizer = None
    try:
        trainer, model, tokenizer = build_training_model(nick, spec["target_modules"])
        vocab_size = model.config.vocab_size
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
        max_L = probe_max_length(model, optimizer, vocab_size)
        recommended = int(max_L * SAFETY_MARGIN) // 8 * 8
        results[nick] = {"vocab": vocab_size, "max_L": max_L, "recommended": recommended}
        print(f"{nick}: vocab={vocab_size} max_fit_L={max_L} -> recommended budget={recommended}")
    finally:
        del trainer, model, tokenizer, optimizer
        gc.collect()
        torch.cuda.empty_cache()

print("\n\n# Copy into core/training/packing_budgets.py -> PACKING_BUDGETS:")
for nick, r in results.items():
    print(f'    "{nick}": {r["recommended"]},  # vocab={r["vocab"]}, max_fit_L={r["max_L"]}')


=== qwen_3b ===


You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2026-06-24 11:35:09.803 | INFO     | core.training.row_scoped_embedding_training:install_row_scoped_grad:65 - row_scoped_embedding: new_ids=[151665, 151666]; tied=True; vocab=151936; hidden=2048
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
The input hidden states seems to be silently casted in float32, this might be related to the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in torch.bfloat16.


  warmup L=4096: OK peak=21.5GiB
  ramp   L=8192: OK peak=33.2GiB
  ramp   L=16384: OK peak=51.8GiB
  ramp   L=32768: OOM
  bisect L=24576: OK peak=70.4GiB
  bisect L=28672: OOM
  bisect L=26624: OK peak=75.0GiB
  bisect L=27648: OOM
  bisect L=27136: OOM
  bisect L=26880: OOM
qwen_3b: vocab=151936 max_fit_L=26624 -> recommended budget=23960

=== llama_3b ===


...

2026-06-24 11:35:46.342 | INFO     | core.training.row_scoped_embedding_training:install_row_scoped_grad:65 - row_scoped_embedding: new_ids=[128256, 128257]; tied=True; vocab=128320; hidden=3072
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  warmup L=4096: OK peak=21.1GiB
  ramp   L=8192: OK peak=32.3GiB
  ramp   L=16384: OK peak=48.8GiB
  ramp   L=32768: OOM
  bisect L=24576: OK peak=65.3GiB
  bisect L=28672: OK peak=73.5GiB
  bisect L=30720: OOM
  bisect L=29696: OOM
  bisect L=29184: OK peak=74.6GiB
  bisect L=29440: OK peak=75.1GiB
llama_3b: vocab=128320 max_fit_L=29440 -> recommended budget=26496

=== phi4_mini ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2026-06-24 11:36:35.951 | INFO     | core.training.row_scoped_embedding_training:install_row_scoped_grad:65 - row_scoped_embedding: new_ids=[200029, 200030]; tied=True; vocab=200064; hidden=3072
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  warmup L=4096: OK peak=27.8GiB
  ramp   L=8192: OK peak=44.7GiB
  ramp   L=16384: OK peak=69.2GiB
  ramp   L=32768: OOM
  bisect L=24576: OOM
  bisect L=20480: OOM
  bisect L=18432: OOM
  bisect L=17408: OK peak=72.3GiB
  bisect L=17920: OK peak=73.8GiB
  bisect L=18176: OK peak=74.6GiB
phi4_mini: vocab=200064 max_fit_L=18176 -> recommended budget=16352


# Copy into core/training/packing_budgets.py -> PACKING_BUDGETS:
    "qwen_3b": 23960,  # vocab=151936, max_fit_L=26624
    "llama_3b": 26496,  # vocab=128320, max_fit_L=29440
    "phi4_mini": 16352,  # vocab=200064, max_fit_L=18176


## Fill in the budgets

Copy the printed values into `PACKING_BUDGETS` in `src/core/training/packing_budgets.py`, replacing the `0  # TODO` placeholders. Sanity checks:

- Each `recommended` budget should be **well above the dataset's longest chain** (~24k for the `middle_truncated24000` splits) — otherwise the trainer's `budget >= max length` assertion will fire.
- `phi4_mini` (200k vocab) should land **lower** than `qwen_3b` / `llama_3b`, since its logits tensor `[L, vocab]` is the largest.

The probe runs eager; the 10% `SAFETY_MARGIN` covers the `torch_compile` memory delta and fragmentation. If a real run still OOMs, lower the affected budget and re-run.